In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

import ipywidgets as widgets
from IPython.display import display

In [2]:
hotel_df = pd.read_csv("../artifacts/raw_data/train.csv")
hotel_df.drop(columns=["Unnamed: 0", "Booking_ID"], inplace=True)
hotel_df.head()

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,2,0,0,3,Meal Plan 1,0,Room_Type 4,56,2018,6,21,Online,0,0,0,87.32,1,Not_Canceled
1,2,0,0,1,Meal Plan 1,0,Room_Type 1,159,2018,2,13,Offline,0,0,0,70.00,0,Not_Canceled
2,2,0,0,2,Not Selected,0,Room_Type 1,86,2018,8,12,Online,0,0,0,107.10,1,Not_Canceled
3,2,0,2,3,Not Selected,0,Room_Type 1,116,2018,6,30,Online,0,0,0,75.27,0,Not_Canceled
4,2,0,1,3,Meal Plan 1,0,Room_Type 1,243,2018,11,24,Online,0,0,0,73.95,0,Canceled


# Cleanliness Check

In [3]:
hotel_df.describe()

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,required_car_parking_space,lead_time,arrival_year,arrival_month,arrival_date,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests
count,29020.000000,29020.000000,29020.000000,29020.000000,29020.000000,29020.000000,29020.000000,29020.000000,29020.000000,29020.000000,29020.000000,29020.000000,29020.000000,29020.000000
mean,1.845279,0.105100,0.817402,2.206892,0.031185,85.394383,2017.821330,7.419538,15.620227,0.026017,0.023225,0.156720,103.336179,0.622708
std,0.517494,0.403363,0.873181,1.419246,0.173821,85.979604,0.383082,3.070393,8.747763,0.159187,0.361599,1.772564,35.096651,0.785522
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2017.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,0.000000,0.000000,1.000000,0.000000,17.000000,2018.000000,5.000000,8.000000,0.000000,0.000000,0.000000,80.300000,0.000000
50%,2.000000,0.000000,1.000000,2.000000,0.000000,57.000000,2018.000000,8.000000,16.000000,0.000000,0.000000,0.000000,99.450000,0.000000
75%,2.000000,0.000000,2.000000,3.000000,0.000000,126.000000,2018.000000,10.000000,23.000000,0.000000,0.000000,0.000000,120.000000,1.000000
max,4.000000,10.000000,7.000000,17.000000,1.000000,443.000000,2018.000000,12.000000,31.000000,1.000000,13.000000,58.000000,540.000000,5.000000


In [4]:
hotel_df.isnull().sum()

no_of_adults                            0
no_of_children                          0
no_of_weekend_nights                    0
no_of_week_nights                       0
type_of_meal_plan                       0
required_car_parking_space              0
room_type_reserved                      0
lead_time                               0
arrival_year                            0
arrival_month                           0
arrival_date                            0
market_segment_type                     0
repeated_guest                          0
no_of_previous_cancellations            0
no_of_previous_bookings_not_canceled    0
avg_price_per_room                      0
no_of_special_requests                  0
booking_status                          0
dtype: int64

In [5]:
hotel_df.duplicated().sum()

np.int64(7689)

In [6]:
hotel_df.drop_duplicates(inplace=True)
hotel_df.shape

(21331, 18)

# Analyze Cols

## Cat vars and few unique numerical vars

In [70]:
low_var_cols = ['no_of_adults', 
                           'no_of_children', 
                           'no_of_weekend_nights', 
                           'no_of_week_nights', 
                           'type_of_meal_plan',
                           'required_car_parking_space',
                           'arrival_year',
                           'arrival_month',
                           'market_segment_type',
                           'repeated_guest', 
                           'room_type_reserved', 
                           'no_of_previous_cancellations',
                           'no_of_previous_bookings_not_canceled',
                           'no_of_special_requests',
                           'booking_status']
dropdown = widgets.Dropdown(
    options=low_var_cols,
    description="Select a value",
    value=None
)

def bar_plot_variables(col='no_of_adults'):
    try:
        column = col.new 
    except AttributeError:
        column = col
    vals_dict = hotel_df[column].value_counts().to_dict()
    vals_dict = dict(sorted(vals_dict.items(), key=lambda x: x[0]))
    fig = px.bar(x=vals_dict.keys(), 
                         y=vals_dict.values(), 
                         title=" ".join(column.split("_")).title(),
                         labels = {"x": " ".join(column.split("_")).title(),
                                   "y": "Count"}
    )
    fig.show()
dropdown.observe(bar_plot_variables, names='value')

dropdown

Dropdown(description='Select a value', options=('no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_…

Data is imbalanced

## Many unique cat vars

In [ ]:
many_var_cols = hotel_df.columns.difference(low_var_cols)
dropdown = widgets.Dropdown(
    options=many_var_cols,
    description="Select a value",
    value=None
)

def histogram_variables(col='arrival_date'):
    try:
        column = col.new 
    except AttributeError:
        column = col
    fig = px.histogram(hotel_df, 
                               column,
                               title=" ".join(column.split("_")).title(),
                               labels = {"x": " ".join(column.split("_")).title(),
                                         "y": "Count"}
    )
    fig.show()
dropdown.observe(histogram_variables, names='value')

dropdown

Dropdown(description='Select a value', options=('arrival_date', 'avg_price_per_room', 'lead_time'), value=None…

In [65]:
cat_cols = [
    'type_of_meal_plan',
    'required_car_parking_space',
    'room_type_reserved',
    'market_segment_type',
    'repeated_guest',
    'booking_status'
]
num_cols = hotel_df.columns.difference(cat_cols)
num_cols

Index(['arrival_date', 'arrival_month', 'arrival_year', 'avg_price_per_room',
       'lead_time', 'no_of_adults', 'no_of_children',
       'no_of_previous_bookings_not_canceled', 'no_of_previous_cancellations',
       'no_of_special_requests', 'no_of_week_nights', 'no_of_weekend_nights'],
      dtype='object')

In [ ]:
def plot_bivatiate_num(df, target, num_features):
    num_plots = len(num_features)
    num_rows = (num_plots+1)//2

    fig, axes = plt.subplots(num_rows, 2, figsize=(15, num_rows*5))
    axes = axes.flatten()

    for i, column in enumerate(num_features):
        sns.boxplot(x=target, y=column, ax=axes[i], data=df, palette="Blues")
        axes[i]. set_title(f"{column} VS {target}")
    
    plt.tight_layout()
    plt.show()

12